<a href="https://colab.research.google.com/github/alperarslan19/asthma-voice-dl/blob/main/notebooks/24_phase3_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/alperarslan19/asthma-voice-dl/blob/main/notebooks/24_phase3_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 24 — Faz 3 hazırlığı: gün-gruplu split'ler, dondurulmuş baz çizgisi, G1 ve D-035 seçimi (CPU)

**Ne yapıyor, neden?** (D-041, D-042; `docs/PHASE3_DESIGN.md`)
1. **Gün-gruplu split'ler:** aynı kayıt gününün bütün katılımcıları ya eğitimde ya testte olur. Böylece model "bu gün kaydedilenler hep hasta" gibi bir günün akustik izini öğrenip ondan yararlanamaz.
2. **Dondurulmuş problar bu split'lerde (EXP-016DG):** Faz 2'nin aynı probu, yeni split'lerde. İki işe yarar:
   - (a) Faz 3'te fine-tuning ile karşılaştırılacak baz çizgisi;
   - (b) her fold için BEATs / WavLM seçimi (D-035), yalnız o fold'un eğitim katılımcılarıyla.
3. **G1:** dondurulmuş skorlar gün-gruplu split'te de saat, tarih ve yaşın ötesinde etiketle ilişkili mi? Okuma kuralı sonuçlardan önce yazıldı (PHASE3_DESIGN 5.1).

GPU gerekmez: *Runtime → CPU*. Süre ~1–3 saat [tahmin]. Problar kaldığı yerden devam eder; bağlantı koparsa *Run all* yeterli.

## 0. Oturum kurulumu

In [8]:
import shutil, subprocess, platform, sys
print('Python:', sys.version.split()[0], '|', platform.platform())
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                         capture_output=True, text=True).stdout)
else:
    print('UYARI: GPU yok (CPU oturumu). Bu notebook CPU üzerinde çalışır; sorun değil.')

Python: 3.13.16 | Linux-6.6.122+-x86_64-with-glibc2.39
UYARI: GPU yok (CPU oturumu). Bu notebook CPU üzerinde çalışır; sorun değil.


In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
from pathlib import Path
import subprocess, sys, os, json, shutil, hashlib, codecs, platform
DRIVE_ROOT  = Path('/content/drive/MyDrive/asthma-voice')
DERIVED     = DRIVE_ROOT / 'data_derived'
MODELS      = DRIVE_ROOT / 'models'
EXPERIMENTS = DRIVE_ROOT / 'experiments'
DRIVE_CACHE = DERIVED / 'audio_cache_v1'
LOCAL_CACHE = Path('/content/audio_cache_v1')
EMB_DIR     = DERIVED / 'embeddings_v1'
GITHUB_USER, REPO_NAME = 'alperarslan19', 'asthma-voice-dl'
REPO_DIR    = Path('/content') / REPO_NAME
BACKBONES   = ['cnn10', 'cnn14', 'cnn14_16k', 'beats', 'wavlm_base_plus', 'wavlm_large']
EXPECTED_N_RECORDINGS, EXPECTED_N_PARTICIPANTS = 2393, 342   # Faz 1 önbellek kohortu (reports/audio_cache)
for p in [DERIVED / 'participant_context.csv', DERIVED / 'splits' / 'outer_r0.csv', DRIVE_CACHE / 'cache_info.json']:
    assert p.exists(), f'Bulunamadı: {p}'
for p in [MODELS, EXPERIMENTS, EMB_DIR]:
    p.mkdir(parents=True, exist_ok=True)


def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


def sh(cmd, check=True):
    """Kabuk komutunu çalıştırır, çıktıyı canlı yazar. check=True iken komut başarısızsa hücreyi DURDURUR.
    (`!komut` başarısız olsa da hücre devam eder; 'Run all' hatayı fark etmeden sonraki adıma geçerdi.)"""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    dec = codecs.getincrementaldecoder('utf-8')('replace')
    while True:
        b = p.stdout.read1(1 << 14)
        if not b:
            break
        sys.stdout.write(dec.decode(b))
        sys.stdout.flush()
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f'Komut başarısız (çıkış kodu {rc}): {cmd[:300]}')
    return None if check else rc


print('Girdiler mevcut.')

Girdiler mevcut.


In [11]:
# Repo: yoksa klonla; varsa GitHub'daki main'e güvenli biçimde geç (önceki çalıştırmanın rapor dalında kalmış olabilir)
from google.colab import userdata
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception as e:
    raise RuntimeError("GITHUB_TOKEN okunamadı: sol paneldeki anahtar (Secrets) simgesinden GITHUB_TOKEN ekli mi ve "
                       "'Notebook access' açık mı?") from e
assert token, 'GITHUB_TOKEN boş'
REMOTE = f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git'


def git(*args, check=True, show=False):
    """git komutu; çıktıda token gizlenir. check=True iken hata durumunda hücreyi durdurur."""
    r = subprocess.run(['git', '-C', str(REPO_DIR), *args], capture_output=True, text=True)
    r.stdout, r.stderr = r.stdout.replace(token, '***'), r.stderr.replace(token, '***')
    if show and (r.stdout + r.stderr).strip():
        print((r.stdout + r.stderr).strip())
    if check and r.returncode != 0:
        cmd = ' '.join(args[:3]).replace(token, '***')
        raise RuntimeError(f'git {cmd} başarısız (çıkış {r.returncode}):\n{(r.stdout + r.stderr).strip()}')
    return r


if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
    raise RuntimeError(f'{REPO_DIR} var ama git deposu değil (yarım kalmış klon?). Şunu çalıştırıp bu hücreyi yeniden '
                       f'çalıştır:  !rm -rf {REPO_DIR}')
if not REPO_DIR.exists():
    r = subprocess.run(['git', 'clone', REMOTE, str(REPO_DIR)], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('git clone başarısız:\n' + (r.stdout + r.stderr).replace(token, '***'))
    print('Klonlandı.')
else:
    git('remote', 'set-url', 'origin', REMOTE)            # token yenilendiyse
    git('fetch', '--prune', 'origin')
    branch = git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip()
    unpushed = git('log', 'HEAD', '--not', '--remotes', '--oneline').stdout.strip()
    dirty = git('status', '--porcelain', '--untracked-files=all').stdout.strip()
    if unpushed:
        raise RuntimeError(
            f"'{branch}' dalında GitHub'a gönderilmemiş commit var:\n{unpushed}\n"
            "Bu, yarım kalmış bir rapor gönderimidir; raporu kaybetmemek için önce 'Raporu GitHub'a gönder' hücresini "
            "çalıştır (aynı commit'i yeniden gönderir), sonra bu hücreyi yeniden çalıştır.")
    if dirty:
        at_main = branch == 'main' and git('rev-parse', 'HEAD').stdout == git('rev-parse', 'origin/main').stdout
        if not at_main:
            raise RuntimeError(
                f"Repo klasöründe commit edilmemiş dosyalar var (dal: {branch}):\n{dirty}\n"
                "Bunlar önceki bir çalıştırmanın raporlarıysa önce gönder hücresini çalıştır. Gerekmiyorsa klasörü silip "
                f"bu hücreyi yeniden çalıştır:  !rm -rf {REPO_DIR}")
        print('UYARI: main zaten GitHub ile aynı; önceki çalıştırmadan kalan commit edilmemiş dosyalara dokunulmadı:\n' + dirty)
    else:
        git('checkout', '-B', 'main', 'origin/main')       # temiz klasör: main = GitHub'daki main (yerel diğer dallar kalır)
        if branch != 'main':
            print(f"'{branch}' dalından main'e geçildi (o dal GitHub'da duruyor).")
assert git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip() == 'main'
COMMIT = git('rev-parse', 'HEAD').stdout.strip()
assert COMMIT == git('rev-parse', 'origin/main').stdout.strip(), 'yerel main GitHub main ile aynı değil'
print('Dal: main | commit:', COMMIT[:12], '(= origin/main)')

Dal: main | commit: 15a0cbbd3c1d (= origin/main)


In [12]:
sh(f'pip install -q -r {REPO_DIR}/requirements-colab.txt')
import numpy as np, pandas as pd, sklearn, statsmodels
print('numpy', np.__version__, '| pandas', pd.__version__, '| sklearn', sklearn.__version__, '| statsmodels', statsmodels.__version__)

numpy 2.1.3 | pandas 2.2.3 | sklearn 1.6.1 | statsmodels 0.15.0


In [13]:
# Kapı: tasarım ve kod main'de; girdiler Drive'da
for p in ['docs/PHASE3_DESIGN.md', 'scripts/make_splits.py', 'scripts/run_probes.py', 'scripts/evaluate_phase3.py',
          'reports/context/EVAL-016_context.json']:
    if git('ls-files', '--error-unmatch', p, check=False).returncode != 0:
        raise RuntimeError(f'{p} main dalında yok: Faz 3 PR\'ı birleştirilmiş mi? Sonra 0. bölümü yeniden çalıştır.')
dec = (REPO_DIR / 'DECISIONS.md').read_text()
assert '## D-041' in dec and '## D-042' in dec, 'D-041 / D-042 main dalında yok'
RECMAP = DERIVED / 'recording_map.csv'
MFCC = DERIVED / 'mfcc' / 'features_cache.csv'
assert RECMAP.exists(), f'{RECMAP} yok (Faz 1 ses denetiminin çıktısı)'
assert MFCC.exists(), 'EXP-011 özellik dosyası (features_cache.csv) yok'
for b in BACKBONES:
    assert (EMB_DIR / b / 'MANIFEST.sha256').exists(), f'{b}: gömmeler yok (notebook 21)'
SPLITS_DAY = DERIVED / 'splits_day'
PROBE_DAY = EXPERIMENTS / 'phase3' / 'frozen_probe_day'
print('Kapı geçti.')

Kapı geçti.


## 1. Gün-gruplu split'ler (3 tekrar × 5 fold)
Katılımcı kimliği içermeyen manifest `reports/splits_day/` altına yazılır. Split üretimi belirlenimcidir: yeniden çalıştırmak aynı dosyaları üretir.

In [14]:
sh(f'cd {REPO_DIR} && python scripts/make_splits.py --context {DERIVED}/participant_context.csv --recording-map {RECMAP} '
   f'--out-dir {SPLITS_DAY} --report-dir {REPO_DIR}/reports/splits_day --n-repeats 3 --group-by-day')

Kohort: 342 katılımcı (283 astım / 59 sağlıklı); son sağlıklı kayıt günü 2024-04-02
Tabakalar (birleştirmeden sonra): {'0|early|<=40': 32, '0|early|>40': 27, '1|early|<=40': 31, '1|early|>40': 67, '1|late|<=40': 53, '1|late|>40': 132}
Birleştirilen seyrek tabakalar: [{'stratum': '0|early|unknown', 'n': 1, 'merged_into': '0|early|<=40'}, {'stratum': '0|unknown|unknown', 'n': 1, 'merged_into': '0|early|<=40'}]
r=0: outer_r0.csv sha256=d46c8875fb87…  test sağlıklı/fold = [13, 12, 12, 11, 11]  iç doğrulama sağlıklı = [10, 10, 10, 10, 10]
r=1: outer_r1.csv sha256=70753773c40c…  test sağlıklı/fold = [13, 12, 12, 11, 11]  iç doğrulama sağlıklı = [9, 9, 9, 11, 10]
r=2: outer_r2.csv sha256=07179c37b36f…  test sağlıklı/fold = [12, 12, 12, 12, 11]  iç doğrulama sağlıklı = [10, 10, 10, 10, 10]
Manifest: /content/asthma-voice-dl/reports/splits_day/splits_manifest.md


## 2. Dondurulmuş problar gün-gruplu split'lerde (EXP-016DG)
En uzun adım. Her (kol, tekrar) bitince Drive'a kaydedilir; yeniden çalıştırmada bitenler atlanır.

In [15]:
sh(f'cd {REPO_DIR} && python scripts/run_probes.py --exp EXP-016DG --emb-root {EMB_DIR} --mfcc-features {MFCC} '
   f'--context {DERIVED}/participant_context.csv --splits-dir {SPLITS_DAY} --out-dir {PROBE_DAY} '
   f'--report-dir {REPO_DIR}/reports/phase3 --n-repeats 3')
sel = pd.read_csv(PROBE_DAY / 'd035_selection.csv')
assert len(sel) == 15 and set(sel.selected) <= {'beats', 'wavlm_base_plus', 'wavlm_large'}, 'D-035 seçimi eksik'
print('D-035 fold başına seçim (yalnız iç doğrulama):', sel.selected.value_counts().to_dict())

  kol cnn10: kayıt/görev {1: 342, 2: 342, 3: 342, 4: 341, 5: 342, 6: 342, 7: 342}; boyut 512
  kol cnn14: kayıt/görev {1: 342, 2: 342, 3: 342, 4: 341, 5: 342, 6: 342, 7: 342}; boyut 2048
  kol cnn14_16k: kayıt/görev {1: 342, 2: 342, 3: 342, 4: 341, 5: 342, 6: 342, 7: 342}; boyut 2048
  kol beats: kayıt/görev {1: 342, 2: 342, 3: 342, 4: 341, 5: 342, 6: 342, 7: 342}; boyut 9216
  kol wavlm_base_plus: kayıt/görev {1: 342, 2: 342, 3: 342, 4: 341, 5: 342, 6: 342, 7: 342}; boyut 9216
  kol wavlm_large: kayıt/görev {1: 342, 2: 342, 3: 342, 4: 341, 5: 342, 6: 342, 7: 342}; boyut 24576
  kol mfcc_lr: kayıt/görev {1: 342, 2: 342, 3: 342, 4: 341, 5: 342, 6: 342, 7: 342}; boyut 44
  kol mfcc_mlp: kayıt/görev {1: 342, 2: 342, 3: 342, 4: 341, 5: 342, 6: 342, 7: 342}; boyut 44
EXP-016DG: 8 kol × 3 tekrar × 5 fold × 7 görev
  cnn10 r0: 24 s; füzyon AUC (havuz) 0.825
  cnn14 r0: 39 s; füzyon AUC (havuz) 0.752
  cnn14_16k r0: 38 s; füzyon AUC (havuz) 0.696
  beats r0: 38 s; füzyon AUC (havuz) 0.896
  wa

## 3. G1: gün-gruplu skorlarla T1
Okuma kuralı sonuçlardan önce yazıldı. Tablo ayrıca Faz 3'ün dondurulmuş baz çizgisidir.

In [16]:
sh(f'cd {REPO_DIR} && python scripts/evaluate_phase3.py frozen-dg --probe-dir {PROBE_DAY} --splits-dir {SPLITS_DAY} '
   f'--context {DERIVED}/participant_context.csv --report-dir {REPO_DIR}/reports/phase3 --out-dir {EXPERIMENTS}/phase3 '
   f'--stage1-report {REPO_DIR}/reports/context/EVAL-016_context.json')

  cnn10: p 0.003863 OR 2.29 CV3 0.0823 E1 0.823
  cnn14: p 0.008209 OR 2.01 CV3 0.0257 E1 0.767
  cnn14_16k: p 0.2659 OR 1.43 CV3 0.349 E1 0.698
  beats: p 0.0002102 OR 3.30 CV3 0.0353 E1 0.884
  wavlm_base_plus: p 0.002091 OR 2.54 CV3 0.12 E1 0.864
  wavlm_large: p 0.0001474 OR 3.44 CV3 0.0542 E1 0.884
  mfcc_lr: p 0.04047 OR 1.75 CV3 0.271 E1 0.718
  mfcc_mlp: p 0.04881 OR 1.78 CV3 0.445 E1 0.729
# G1 — dondurulmuş problar gün-gruplu split'lerde (EXP-016DG) ve T1 (otomatik üretildi; katılımcı ID'si içermez)

**Okuma kuralı (önceden yazıldı):** ŝ_DG ile T1-PLR p < 0.05 ve OR > 1 → gün parmak izi yoluyla açıklanmıyor (PHASE3_DESIGN 5.1). CV3 ve Holm betimsel. Kohort 339 (282 / 57); 3 tekrar × 5 fold, gün-gruplu.

| kol | E1 fold AUC (gün-gruplu) | T1 PLR p | OR / SD [bootstrap %95] | CV3 p | Holm (betimsel) | Aşama 1 (katılımcı-gruplu): p · OR | G1 okuması |
|---|---|---|---|---|---|---|---|
| cnn10 | 0.823 ± 0.093 | 0.004 | 2.293 [1.399, 7.168] | 0.082 | 0.012 | <0.001 · 3.064 | gün p

## 4. Raporları GitHub'a gönder (yeni dal)
Yalnız agrega raporlar: `reports/splits_day/`, `reports/phase3/`.

In [17]:
# Rapor commit'i her seferinde YENİ, zaman damgalı bir dala gider (aynı adlı dala ikinci gönderim reddedilirdi).
# Gönderim başarısız olursa commit yerel dalda kalır; bu hücreyi yeniden çalıştırmak aynı commit'i yeniden gönderir.
import re, time
import pandas as pd
BASE_BRANCH, PATHS, MSG = 'phase3-prep', ['reports/splits_day', 'reports/phase3'], 'Faz 3 hazırlığı: gün-gruplu split, EXP-016DG, G1, D-035 seçimi (Colab)'
git('config', 'user.name', GITHUB_USER)
git('config', 'user.email', f'{GITHUB_USER}@users.noreply.github.com')
cur = git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip()
if cur.startswith(BASE_BRANCH + '-') and git('log', 'HEAD', '--not', '--remotes', '--oneline').stdout.strip():
    BRANCH = cur
    print('Önceki denemenin commit\'i yeniden gönderiliyor:', BRANCH)
else:
    assert cur == 'main', f"Beklenmeyen dal '{cur}': 0. bölümdeki repo hücresini yeniden çalıştır."
    git('add', '--', *PATHS)
    staged = git('diff', '--cached', '--name-only').stdout.split()
    assert staged, 'Gönderilecek değişiklik yok (rapor üretilmedi mi? yukarıdaki hücrenin çıktısına bak).'
    bad = [s for s in staged if not any(s == p or s.startswith(p.rstrip('/') + '/') for p in PATHS)]
    assert not bad, f'Beklenmeyen dosya: {bad}'
    ids = set(pd.read_csv(DERIVED / 'participant_context.csv').participant_id.astype(int))   # D-014 son kontrol
    for s in staged:
        txt = (REPO_DIR / s).read_text(errors='ignore')
        found = {int(m) for m in re.findall(r'(?<![\d.])(\d{6})(?![\d.])', txt)} & ids
        assert token not in txt and not found, f'{s}: token ya da katılımcı ID\'si içeriyor → commit edilmedi (D-014)'
    BRANCH = f"{BASE_BRANCH}-{time.strftime('%Y%m%d-%H%M%S', time.gmtime())}"
    git('checkout', '-b', BRANCH)
    git('commit', '-m', MSG)
    print('commit:', git('rev-parse', '--short', 'HEAD').stdout.strip(), '| dosyalar:', staged)
r = git('push', '-u', 'origin', BRANCH, check=False, show=True)
if r.returncode != 0:
    raise RuntimeError(f"Push başarısız. Commit yerel '{BRANCH}' dalında duruyor (kaybolmadı). Token/ağ sorununu giderip "
                       "BU hücreyi yeniden çalıştır; aynı commit yeniden gönderilir.")
git('checkout', 'main')
print(f'Gönderildi: {BRANCH}\nPR aç: https://github.com/{GITHUB_USER}/{REPO_NAME}/pull/new/{BRANCH}')

commit: 99fb6f4 | dosyalar: ['reports/phase3/EXP-016DG_frozen.json', 'reports/phase3/EXP-016DG_frozen.md', 'reports/phase3/G1_frozen_daygrouped.json', 'reports/phase3/G1_frozen_daygrouped.md', 'reports/splits_day/splits_manifest.json', 'reports/splits_day/splits_manifest.md']
branch 'phase3-prep-20261011-084815' set up to track 'origin/phase3-prep-20261011-084815'.
remote: 
remote: Create a pull request for 'phase3-prep-20261011-084815' on GitHub by visiting:        
remote:      https://github.com/alperarslan19/asthma-voice-dl/pull/new/phase3-prep-20261011-084815        
remote: 
To https://github.com/alperarslan19/asthma-voice-dl.git
 * [new branch]      phase3-prep-20261011-084815 -> phase3-prep-20261011-084815
Gönderildi: phase3-prep-20261011-084815
PR aç: https://github.com/alperarslan19/asthma-voice-dl/pull/new/phase3-prep-20261011-084815
